In [9]:
import xarray as xr
from pymodis.convertmodis_gdal import convertModisGDAL
import os
import numpy as np
from osgeo import gdal
import rioxarray
from satpy import Scene
import matplotlib.pyplot as plt
import autoroot
from rs_tools._src.geoprocessing.modis.rescale import convert_integers2radiances
from mpl_toolkits.basemap import Basemap, cm
from scipy.interpolate import griddata
import rasterio

In [10]:
def interp_coords(lat: np.array, lon: np.array, desired_size: tuple=(2030, 1354)) -> np.array:
    # Create grid coordinates for the original and interpolated arrays
    original_rows = np.stack((np.linspace(0, lat.shape[0], num=lat.shape[0]),) * lat.shape[1], axis=1)
    original_cols = np.stack((np.linspace(0, lat.shape[1], num=lat.shape[1]),) * lat.shape[0], axis=0)
    sf_x = desired_size[0]/lat.shape[0]
    sf_y = desired_size[1]/lat.shape[1]
    original_rows = (original_rows) * sf_x
    original_cols = (original_cols) * sf_y
    interp_rows, interp_cols = np.indices(desired_size)

    # Flatten the original array and grid coordinates
    original_positions = np.column_stack((original_rows.flatten(), original_cols.flatten()))
    original_lat = lat.values.flatten()
    original_lon = lon.values.flatten()

    interp_positions = np.column_stack((interp_rows.flatten(), interp_cols.flatten()))

    # Perform 2D interpolation using griddata
    interp_lat = griddata(original_positions, original_lat, interp_positions, method='cubic')
    interp_lon = griddata(original_positions, original_lon, interp_positions, method='cubic')

    # Reshape the interpolated array to the desired size
    interp_lat = interp_lat.reshape(desired_size)
    interp_lon = interp_lon.reshape(desired_size)

    return interp_lat, interp_lon
def get_corner_coords(lat_0, lon_0, lat, lon):
    m1 = Basemap(projection='ortho',lon_0=lon_0,lat_0=lat_0,resolution=None)

    xpt0, ypt0 = m1(lon_0,lat_0) # x/y map projection coordinates (in meters)

    xpt1, ypt1 = m1(lon[0,0],lat[0,0]) 
    xpt2, ypt2 = m1(lon[0,-1],lat[0,-1]) 
    xpt3, ypt3 = m1(lon[-1,-1], \
                    lat[-1,-1])
    xpt4, ypt4 = m1(lon[-1,0],lat[-1,0])

    llx = min(xpt1,xpt2,xpt3,xpt4) - xpt0  # lower left
    lly = min(ypt1,ypt2,ypt3,ypt4) - ypt0

    urx = max(xpt1,xpt2,xpt3,xpt4) - xpt0  # upper right
    ury = max(ypt1,ypt2,ypt3,ypt4) - ypt0

    return (llx, lly, urx, ury)
def get_central_coords(lat, lon):

    lat_max = lat[0,0]
    lat_min = lat[-1,-1]

    lat_0 = lat_min + (lat_max - lat_min) / 2.

    long_min = min(lon[0,0],lon[-1,-1])
    long_max = max(lon[0,0],lon[-1,-1])

    lon_0 = long_min + (long_max - long_min) / 2.

    return lat_0, lon_0

In [11]:
ds = xr.open_dataset('/mnt/data8tb/fire_detection/modis/L1b/MOD021KM.A2023219.0835.061.2023220144500.hdf', engine='netcdf4')
ds_fire = xr.open_dataset('/mnt/data8tb/fire_detection/modis/AF/MOD14.A2023219.0835.061.2023220113415.hdf', engine='netcdf4')

In [12]:
fire_mask = ds_fire['fire mask'].values

In [13]:
fire_mask.shape

(2030, 1354)

In [14]:
lat = ds.Latitude
lon = ds.Longitude

data = ds.EV_1KM_Emissive
corrected_data = convert_integers2radiances(data)

along_track = corrected_data.shape[0]
cross_trak = corrected_data.shape[1]

In [16]:
lat_max = lat[0,0]
lat_min = lat[-1,-1]

lat_0 = lat_min + (lat_max - lat_min) / 2.

long_min = min(lon[0,0],lon[-1,-1])
long_max = max(lon[0,0],lon[-1,-1])

lon_0 = long_min + (long_max - long_min) / 2.

In [17]:
m1 = Basemap(projection='ortho',lon_0=lon_0,lat_0=lat_0,resolution=None)

xpt0, ypt0 = m1(lon_0,lat_0) # x/y map projection coordinates (in meters)

xpt1, ypt1 = m1(lon[0,0],lat[0,0]) 
xpt2, ypt2 = m1(lon[0,-1],lat[0,-1]) 
xpt3, ypt3 = m1(lon[-1,-1], \
                lat[-1,-1])
xpt4, ypt4 = m1(lon[-1,0],lat[-1,0])

llx = min(xpt1,xpt2,xpt3,xpt4) - xpt0  # lower left
lly = min(ypt1,ypt2,ypt3,ypt4) - ypt0

urx = max(xpt1,xpt2,xpt3,xpt4) - xpt0  # upper right
ury = max(ypt1,ypt2,ypt3,ypt4) - ypt0

interp_lat, interp_lon = interp_coords(lat, lon)
lat_0, lon_0 = get_central_coords(interp_lat, interp_lon)
llx, lly, urx, ury = get_corner_coords(lat_0, lon_0, interp_lat, interp_lon)

In [25]:
from rasterio.transform import from_bounds

# Flatten the coordinates and fire data
points = np.array([interp_lon.flatten(), interp_lat.flatten()]).T
values = fire_mask.flatten()

# Create a regular grid for interpolation
num_x, num_y = 1000, 1000  # Resolution of the grid
lon_min, lon_max = np.min(interp_lon), np.max(interp_lon)
lat_min, lat_max = np.min(interp_lat), np.max(interp_lat)

grid_x, grid_y = np.meshgrid(
    np.linspace(lon_min, lon_max, num_x),
    np.linspace(lat_min, lat_max, num_y)
)

# Interpolate the data onto the regular grid
grid_fire = griddata(points, values, (grid_x, grid_y), method='nearest')
grid_fire = np.nan_to_num(grid_fire, nan=0).astype(np.uint8)  # Handle NaNs and enforce integer type

# ** Flip the data to correct the vertical orientation **
grid_fire = np.flipud(grid_fire)

# Define the transform for the new grid
transform = from_bounds(lon_min, lat_min, lon_max, lat_max, num_x, num_y)

# Save the interpolated fire data as GeoTIFF
output_file = "fire_plot_interpolated_corrected_v4.tif"

with rasterio.open(
    output_file,
    'w',
    driver='GTiff',
    height=grid_fire.shape[0],
    width=grid_fire.shape[1],
    count=1,
    dtype=grid_fire.dtype,
    crs="EPSG:4326",
    transform=transform,
) as dst:
    dst.write(grid_fire, 1)

print(f"Saved corrected GeoTIFF to {output_file}")

Saved corrected GeoTIFF to fire_plot_interpolated_corrected_v4.tif
